In [2]:

from langchain_ibm import ChatWatsonx
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser, PydanticOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda

from dotenv import load_dotenv
import os

from langchain_community.document_loaders import PyPDFLoader, CSVLoader, WebBaseLoader, DirectoryLoader
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_ibm import WatsonxEmbeddings
from langchain_chroma import Chroma
from langchain_community.vectorstores import FAISS


from langchain_classic.chains.query_constructor.base import AttributeInfo
from langchain_classic.retrievers import EnsembleRetriever, ContextualCompressionRetriever,BM25Retriever
from langchain_classic.retrievers.self_query.chroma import ChromaTranslator
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_cohere import CohereRerank
from langchain_classic.retrievers.document_compressors import LLMChainExtractor, EmbeddingsFilter, DocumentCompressorPipeline



c:\source2\ollama\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\soldesk\AppData\Local\Temp\ipykernel_14496\3807301162.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, CSVLoader, WebBaseLoader, DirectoryLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
import re
from pydantic import BaseModel, Field
from pprint import pprint

In [4]:

load_dotenv()

apiKey = os.getenv("WATSONX_API_KEY")
project_id = os.getenv("WATSONX_PROJECT_ID")
watsonx_ai_url = os.getenv("WATSONX_URL")
hf_token = os.environ["HF_TOKEN"]
COHERE_API_KEY = os.environ["COHERE_API_KEY"]

watson_embedding = WatsonxEmbeddings(
    model_id="ibm/granite-embedding-278m-multilingual",
    url = f"{watsonx_ai_url}",
    api_key = f"{apiKey}",
    project_id=f"{project_id}"
)

watson_llm = ChatWatsonx(
  model_id="ibm/granite-4-h-small",
  url=f"{watsonx_ai_url}",
  api_key = f"{apiKey}",
  project_id=f"{project_id}",
  max_tokens = 2000,
  params = {
    "temperature":0
  }
)



### pdf load

In [6]:
pdf_path = "./data/2018 서비스·집단 분쟁조정 사례집.pdf"

loader = PyPDFLoader(pdf_path)
docs = loader.load()

docs[10].page_content

'제1장\n일\n반\n분\n쟁\n조\n정\n 사\n례 (\n서\n비\n스 )\n제1장 일반분쟁조정 사례(서비스) ● 3\n사\n례 01 사건번호 2018일나565  | 결정일자 2018. 8. 7.\n세탁 후 갑피 마모 및 경화된 가죽 \n운동화에 대한 손해배상 요구\n주 문\n1. 신청인은 2018. 10. 16.까지 피신청인에게 이 사건 제품(제품명 : ○○○○ 가죽 \n운동화, 색상 : 흰색) 1켤레를 반환한다. \n2. 피신청인은 신청인으로부터 제1항 제품을 반환받음과 동시에 신청인에게 71,000원\n을 지급한다.\n이 유\n1. 기초사실\n가. 신청인은 2017. 6. 6. 가죽 운동화(제품명 : ○○○○ 가죽 운동화, 색상 : 흰색, \n이하 ‘이 사건 제품’) 1켤레를 160,200원에 구매하여 착화하였고, 2018. 1. 10. \n피신청인에게 이 사건 제품의 세탁을 의뢰(세탁비 4,000원)하였는데 수령 후 갑피 \n마모 및 경화된 사실(이하 ‘이 사건 현상’)을 확인하여 피신청인이 재세탁을 하였\n으나, 이후에도 경화현상만 다소 개선될 뿐 갑피 마모 현상이 개선되지 않아 피신\n청인에게 손해배상(세탁비 환급 포함)을 요구하였으며, 피신청인은 세탁과실이 없\n다는 이유로 이를 거부하였다.\n나. 한국소비자원 신발제품심의위원회 심의 결과는 다음과 같다.\n   \n신청인이 주장하는 갑피 벗겨짐(스크래치 등) 증상은 관찰되나 현 제품 상태만\n으로는 제품 훼손의 원인이 세탁 과정상 발생한 것인지 착화 환경에 따른 문제\n인지 단정하기 어려운바, 판단 불가하다.'

### 전처리
- 정규식을 이용한 내용 추출

In [7]:
pattern = r"사\n례\s*\d+.*사건번호.*결정일자.*\d{4}\.\s?\d{1,2}\.\s?\d{1,2}\."

split_text = re.findall(pattern, "".join(docs[10].page_content))
split_text

['사\n례 01 사건번호 2018일나565  | 결정일자 2018. 8. 7.']

In [11]:
# pattern이 존재하면 패턴을 기준으로 문서 분리
if split_text:
    parts = re.split(pattern, "".join(docs[10].page_content))

parts

['제1장\n일\n반\n분\n쟁\n조\n정\n 사\n례 (\n서\n비\n스 )\n제1장 일반분쟁조정 사례(서비스) ● 3\n',
 '\n세탁 후 갑피 마모 및 경화된 가죽 \n운동화에 대한 손해배상 요구\n주 문\n1. 신청인은 2018. 10. 16.까지 피신청인에게 이 사건 제품(제품명 : ○○○○ 가죽 \n운동화, 색상 : 흰색) 1켤레를 반환한다. \n2. 피신청인은 신청인으로부터 제1항 제품을 반환받음과 동시에 신청인에게 71,000원\n을 지급한다.\n이 유\n1. 기초사실\n가. 신청인은 2017. 6. 6. 가죽 운동화(제품명 : ○○○○ 가죽 운동화, 색상 : 흰색, \n이하 ‘이 사건 제품’) 1켤레를 160,200원에 구매하여 착화하였고, 2018. 1. 10. \n피신청인에게 이 사건 제품의 세탁을 의뢰(세탁비 4,000원)하였는데 수령 후 갑피 \n마모 및 경화된 사실(이하 ‘이 사건 현상’)을 확인하여 피신청인이 재세탁을 하였\n으나, 이후에도 경화현상만 다소 개선될 뿐 갑피 마모 현상이 개선되지 않아 피신\n청인에게 손해배상(세탁비 환급 포함)을 요구하였으며, 피신청인은 세탁과실이 없\n다는 이유로 이를 거부하였다.\n나. 한국소비자원 신발제품심의위원회 심의 결과는 다음과 같다.\n   \n신청인이 주장하는 갑피 벗겨짐(스크래치 등) 증상은 관찰되나 현 제품 상태만\n으로는 제품 훼손의 원인이 세탁 과정상 발생한 것인지 착화 환경에 따른 문제\n인지 단정하기 어려운바, 판단 불가하다.']

In [12]:
# metadata로 사용할 정보 추출
# 사례번호, 사건번호, 결정일자 추출

split_text[0]

# 사례번호
re.findall(r"례\s?(\d+)\s?사건번호", split_text[0])[0]

'01'

In [8]:
# LLM으로 사건번호, 결정일자
class CaseMetadata(BaseModel):
    case_number:str = Field(description="사건번호 예: 2018일나565")
    decision_data:str = Field(description="결정일자 예: 2018. 8. 7.")

In [13]:
metadata_prompt = PromptTemplate.from_template(
    """\
다음은 분쟁 조정 사례에 대한 텍스트입니다.\n\n
- case_number: 사건 번호
- decision_date: 결정일자

반드시 JSON 으로 반환하세요
{case_text}
    """
)

structed_llm = watson_llm.with_structured_output(CaseMetadata)
chain = metadata_prompt | structed_llm
case_metadata = chain.invoke({"case_text": split_text[0]})
print(case_metadata)
print(dict(case_metadata))

case_number='2018일나565' decision_data='2018. 8. 7.'
{'case_number': '2018일나565', 'decision_data': '2018. 8. 7.'}


#### page_content 내용 추출

In [14]:
# 주 문 앞 타이틀, 뒤 콘텐츠
parts[1]

# 위치 찾을 때
re.search(r"주 문\n", parts[1]).span()

title = parts[1][:re.search(r"주 문\n", parts[1]).span()[0]].strip()
content = parts[1][re.search(r"주 문\n", parts[1]).span()[0]:].strip()

pprint(title)
pprint(content)

'세탁 후 갑피 마모 및 경화된 가죽 \n운동화에 대한 손해배상 요구'
('주 문\n'
 '1. 신청인은 2018. 10. 16.까지 피신청인에게 이 사건 제품(제품명 : ○○○○ 가죽 \n'
 '운동화, 색상 : 흰색) 1켤레를 반환한다. \n'
 '2. 피신청인은 신청인으로부터 제1항 제품을 반환받음과 동시에 신청인에게 71,000원\n'
 '을 지급한다.\n'
 '이 유\n'
 '1. 기초사실\n'
 '가. 신청인은 2017. 6. 6. 가죽 운동화(제품명 : ○○○○ 가죽 운동화, 색상 : 흰색, \n'
 '이하 ‘이 사건 제품’) 1켤레를 160,200원에 구매하여 착화하였고, 2018. 1. 10. \n'
 '피신청인에게 이 사건 제품의 세탁을 의뢰(세탁비 4,000원)하였는데 수령 후 갑피 \n'
 '마모 및 경화된 사실(이하 ‘이 사건 현상’)을 확인하여 피신청인이 재세탁을 하였\n'
 '으나, 이후에도 경화현상만 다소 개선될 뿐 갑피 마모 현상이 개선되지 않아 피신\n'
 '청인에게 손해배상(세탁비 환급 포함)을 요구하였으며, 피신청인은 세탁과실이 없\n'
 '다는 이유로 이를 거부하였다.\n'
 '나. 한국소비자원 신발제품심의위원회 심의 결과는 다음과 같다.\n'
 '   \n'
 '신청인이 주장하는 갑피 벗겨짐(스크래치 등) 증상은 관찰되나 현 제품 상태만\n'
 '으로는 제품 훼손의 원인이 세탁 과정상 발생한 것인지 착화 환경에 따른 문제\n'
 '인지 단정하기 어려운바, 판단 불가하다.')


In [15]:
# metadata - 사례번호, 사건번호, 결정일자, 제목
# 내용 page_content update

pdf_docs = []
case_metadata = {}

# 사건이 시작되는 페이지 ~ 마지막에서 -2 페이지 까지 반복
for doc in docs[10:-2]:
    split_text = re.findall(pattern, "".join(doc.page_content))
    if split_text:

        # 사례번호 추출
        case_metadata["case_id"] = re.findall(r"례\s?(\d+)\s?사건번호", split_text[0])[0]
    
        # 패턴 기준으로 텍스트 분할
        parts = re.split(pattern, "".join(doc.page_content))
        if re.search(r"주 문\n", parts[1]):
            # 제목 추출
            case_metadata['title'] = parts[1][:re.search(r"주 문\n", parts[1]).span()[0]].replace("\n","").strip()
            # 내용 추출 후 기존 내용 업데이트
            doc.page_content = parts[1][re.search(r"주 문\n", parts[1]).span()[0]:].strip()
        else: 
            case_metadata['title'] = "" 

        i = 0
        while i < 10 :
            try:
                # 사건번호, 결정일자 추출
                response = chain.invoke({"case_text": split_text[0]})
                for k,v in dict(response).items():
                    case_metadata[k] = v.replace("\n","").replace(" ","")
                break
            except:
                i += 1
                continue
        doc.metadata.update(case_metadata)
        pdf_docs.append(doc)   
    else: 
        doc.metadata.update(case_metadata)
        pdf_docs.append(doc)

len(pdf_docs)

188

In [17]:
pdf_docs[10].metadata
pprint(pdf_docs[9].page_content)

('주 문\n'
 '신청인과 피신청인 사이의 이 사건 분쟁조정 신청에 대하여는 조정하지 아니한다.\n'
 '이 유\n'
 '1. 기초사실\n'
 '신청인은 피신청인의 인천-도쿄 왕복항공권 4매(탑승객 : 이○○, 오○○, 이 ∆∆, 이\n'
 '□□, 2017. 12. 18. 10:10 ○○703편 인천 출발, 2017. 12. 21. 17:00 ○○002 편 \n'
 '도쿄 출발)를 628,200원에 구입하였고, 2018. 12. 18. 10:10 출발 예정이던 출국 항\n'
 '공편(이하 ‘이 사건 항공편’이라고 함)이 기상악화로 인하여 지연되어 도착예정시간 \n'
 '12:30보다 4시간 10분 지연된 16:40 도쿄에 도착하였다.\n'
 '2 .  판   단\n'
 '신청인은 이 사건 항공편이 4시간 이상 지연되었고, 이 사건 항공편보다 늦게 출발이 \n'
 '예정되어 있는 항공기(오사카, 나고야행 등)가 먼저 출발하는 것을 확인하였으며, 기상\n'
 '상황으로 인한 연결편 연착에 따른 지연이라는 피신청인의 주장을 납득할 수 없는바, \n'
 '관련 법률에 따른 손해배상을 요구한다.\n'
 '이에 대하여 피신청인은 이 사건 항공편 운송지연의 원인이 당시 인천공항에 강설로 \n'
 '인한 극심한 혼잡과 제빙작업에 의한 것으로, 이는 기상상황으로 인한 지연에 해당하\n'
 '여 「소비자분쟁해결기준」상 손해배상 사유에서 제외되는바, 신청인의 요구를 수용할 \n'
 '수 없다고 주장한다.')


### 2. 분할(split)

In [18]:
# 500 / 50 
chunk_size=500
chunk_overlap=50
splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size,chunk_overlap=chunk_overlap)
split_docs = splitter.split_documents(docs)
print(split_docs[0])

page_content='소비자분쟁조정위원회
2018
서비스·집단 
분쟁조정 사례집' metadata={'producer': 'Acrobat Distiller 9.0.0 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2019-06-05T11:33:24+09:00', 'author': 'PC_A2', 'moddate': '2019-06-05T11:58:31+09:00', 'title': '<32303139303630355FBCD2BAF1C0DABFF820BBE7B7CAC1FD5BBCADBAF1BDBA5D5FB3BBC1F65FC6EDC1FDBABB76657231312E687770>', 'source': './data/2018 서비스·집단 분쟁조정 사례집.pdf', 'total_pages': 200, 'page': 0, 'page_label': '1'}


In [21]:
print(split_docs[1])

page_content='2018 분쟁조정  사례집
소비자 관련 분쟁은 소액이라는 특징 때문에 법원의 소송으로 해결하기 어려운 면이 있습니다. 이러한 점 때문에 
소송 외 분쟁해결기구로서 설립된 소비자분쟁조정위원회는 원만한 합의가 이루어지지 않은 소비자와 사업자에게 
객관적이고 공정한 조정안을 제시함으로써 분쟁이 합리적이고 원활히 해결될 수 있도록 노력하고 있습니다.
소비자기본법에 근거하여 1987년 설립된 소비자분쟁조정위원회는 설립 첫 해 20건의 사건 조정을 시작으로, 2004년 
이후부터는 매년 1,000건이 넘는 사건을 조정하였으며, 2018년에는 3,080여건을 처리하는 등 그 역할을 충실히 
수행하고 있습니다.
특히, 분쟁조정 사건을 신속하고 공정하게 처리하기 위해 2017년에는 소비자기본법 개정을 통해 조정위원을 50명
에서 150명으로 확대하는 등 관련 법·제도 개선으로 소비자권익증진에 기여하고 있습니다.' metadata={'producer': 'Acrobat Distiller 9.0.0 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2019-06-05T11:33:24+09:00', 'author': 'PC_A2', 'moddate': '2019-06-05T11:58:31+09:00', 'title': '<32303139303630355FBCD2BAF1C0DABFF820BBE7B7CAC1FD5BBCADBAF1BDBA5D5FB3BBC1F65FC6EDC1FDBABB76657231312E687770>', 'source': './data/2018 서비스·집단 분쟁조정 사례집.pdf', 'total_pages': 200, 'page': 2, 'page_label': '3'}


In [22]:
# 마침표 뒤에 나오는 줄바꿈 문자는 그대로 두고 나머지 줄바꿈 문자만 제거
text = split_docs[18].page_content
pprint(text)
text = re.sub(r"(?<!\.)\n", " ", text)
text
pprint(text)

('주 문\n'
 '1. 신청인은 2018. 10. 16.까지 피신청인에게 이 사건 제품(제품명 : ○○○○ 가죽 \n'
 '운동화, 색상 : 흰색) 1켤레를 반환한다. \n'
 '2. 피신청인은 신청인으로부터 제1항 제품을 반환받음과 동시에 신청인에게 71,000원\n'
 '을 지급한다.\n'
 '이 유\n'
 '1. 기초사실\n'
 '가. 신청인은 2017. 6. 6. 가죽 운동화(제품명 : ○○○○ 가죽 운동화, 색상 : 흰색, \n'
 '이하 ‘이 사건 제품’) 1켤레를 160,200원에 구매하여 착화하였고, 2018. 1. 10. \n'
 '피신청인에게 이 사건 제품의 세탁을 의뢰(세탁비 4,000원)하였는데 수령 후 갑피 \n'
 '마모 및 경화된 사실(이하 ‘이 사건 현상’)을 확인하여 피신청인이 재세탁을 하였\n'
 '으나, 이후에도 경화현상만 다소 개선될 뿐 갑피 마모 현상이 개선되지 않아 피신\n'
 '청인에게 손해배상(세탁비 환급 포함)을 요구하였으며, 피신청인은 세탁과실이 없\n'
 '다는 이유로 이를 거부하였다.\n'
 '나. 한국소비자원 신발제품심의위원회 심의 결과는 다음과 같다.')
('주 문 1. 신청인은 2018. 10. 16.까지 피신청인에게 이 사건 제품(제품명 : ○○○○ 가죽  운동화, 색상 : 흰색) 1켤레를 '
 '반환한다.  2. 피신청인은 신청인으로부터 제1항 제품을 반환받음과 동시에 신청인에게 71,000원 을 지급한다.\n'
 '이 유 1. 기초사실 가. 신청인은 2017. 6. 6. 가죽 운동화(제품명 : ○○○○ 가죽 운동화, 색상 : 흰색,  이하 ‘이 사건 '
 '제품’) 1켤레를 160,200원에 구매하여 착화하였고, 2018. 1. 10.  피신청인에게 이 사건 제품의 세탁을 의뢰(세탁비 '
 '4,000원)하였는데 수령 후 갑피  마모 및 경화된 사실(이하 ‘이 사건 현상’)을 확인하여 피신청인이 재세탁을 하였 으나, 이후에도 '
 '경화현상만 다소 개선될 뿐 갑피 마모 현상이 개선되지 않아 피신 청

#### indexing - 임베딩

In [19]:
from copy import deepcopy

final_docs =[]
for doc in split_docs:
    new_doc = deepcopy(doc)
    text = re.sub(r"(?<!\.)\n", " ", new_doc.page_content)

    new_doc.page_content = (
        f"### 이 사건은 {new_doc.metadata['title']}에 대한 사례입니다.\n\n"
        f"{text}"
    )

    final_docs.append(new_doc)

In [28]:
len(final_docs)

487

In [20]:
# 로컬저장: chroma
# 메모리: FAISS

vectorstore = Chroma.from_documents(
    documents=final_docs,
    embedding=watson_embedding,
    persist_directory="./db/chroma_db",
    collection_name="customer_disput_cases2"
)

In [21]:
query = "세탁 후 오염에 데한 손해배상은 어떻게 이루어지나요?"
similarity_docs = vectorstore.similarity_search(query,k=5)
pprint(similarity_docs)

[Document(id='8d5ecd5b-3d50-4d5f-abf5-ff4a7408db67', metadata={'total_pages': 200, 'page_label': '12', 'creator': 'PScript5.dll Version 5.2.2', 'producer': 'Acrobat Distiller 9.0.0 (Windows)', 'case_id': '01', 'case_number': '2018일나565', 'page': 11, 'author': 'PC_A2', 'decision_data': '2018.8.7.', 'title': '세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구', 'source': './data/2018 서비스·집단 분쟁조정 사례집.pdf', 'moddate': '2019-06-05T11:58:31+09:00', 'creationdate': '2019-06-05T11:33:24+09:00'}, page_content='### 이 사건은 세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구에 대한 사례입니다.\n\n탁과실로 단정할 수 없다고 판단된 점, 신청인이 착화 과정에서 이 사건 제품을 훼손 하여 그 손해가 발생 및 확대되었을 가능성을 배제할 수 없는 점 등에 비추어 볼 때,  손해의 공평·타당한 분담이라는 손해배상 제도의 지도이념과 상호 양보를 통한 분쟁의  원만한 해결이라는 조정의 취지를 고려하여, 피신청인의 책임을 60%로 제한함이 상당 하다.\n한편, 세탁비와 관련하여,「세탁업 표준약관」제9조 제1항 및 제2항에서는 세탁업자의  책임있는 사유로 세탁물이 손상, 색상변화, 얼룩 등의 하자가 발생였을 때에는 해당  세탁물에 대하여 세탁업자는 고객에게 세탁요금을 청구하지 못하므로, 세탁업자인 피 신청인이 세탁비 4,000원을 신청인에게 환급이 상당하다.\n이상을 종합하면, 신청인은 피신청인에게 이 사건 제품을 반환하고 피신청인은 손해배 상액 67,000원(112,140원 × 60%, 1,000원

In [22]:
for doc in similarity_docs:
    print(doc.metadata['case_id'], doc.metadata['page'])
    print(doc.page_content[:100] + "\n\n")

01 11
### 이 사건은 세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구에 대한 사례입니다.

탁과실로 단정할 수 없다고 판단된 점, 신청인이 착화 과정에서 이 사건 제품


01 11
### 이 사건은 세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구에 대한 사례입니다.

살피건대, 피신청인은 세탁 전부터 이 사건 제품의 상태가 좋지 않았다고 주장


01 10
### 이 사건은 세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구에 대한 사례입니다.

나. 한국소비자원 신발제품심의위원회 심의 결과는 다음과 같다.
    신청인


01 11
### 이 사건은 세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구에 대한 사례입니다.

4 ● 2018 서비스·집단 분쟁조정 사례집 2 .  판   단 신청인은 피


01 10
### 이 사건은 세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구에 대한 사례입니다.

주 문 1. 신청인은 2018. 10. 16.까지 피신청인에게 이 사건 제품




In [23]:
# LangChain 이용하는 방식으로 변경

retriever = vectorstore.as_retriever(search_kwargs={'k':5, "filter":{"case_id":"01"}})
retriever_docs = retriever.invoke(query)
for doc in retriever_docs:
    print(doc.metadata['case_id'], doc.metadata['page'])
    print(doc.page_content[:100] + "\n\n")

01 11
### 이 사건은 세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구에 대한 사례입니다.

탁과실로 단정할 수 없다고 판단된 점, 신청인이 착화 과정에서 이 사건 제품


01 11
### 이 사건은 세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구에 대한 사례입니다.

살피건대, 피신청인은 세탁 전부터 이 사건 제품의 상태가 좋지 않았다고 주장


01 10
### 이 사건은 세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구에 대한 사례입니다.

나. 한국소비자원 신발제품심의위원회 심의 결과는 다음과 같다.
    신청인


01 11
### 이 사건은 세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구에 대한 사례입니다.

4 ● 2018 서비스·집단 분쟁조정 사례집 2 .  판   단 신청인은 피


01 10
### 이 사건은 세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구에 대한 사례입니다.

주 문 1. 신청인은 2018. 10. 16.까지 피신청인에게 이 사건 제품




In [24]:

retriever = vectorstore.as_retriever(search_kwargs={'k':5, "where_document" :{"$contains":"세탁"}})
retriever_docs = retriever.invoke(query)
for doc in retriever_docs:
    print(doc.metadata['case_id'], doc.metadata['page'])
    print(doc.page_content[:100] + "\n\n")

01 11
### 이 사건은 세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구에 대한 사례입니다.

탁과실로 단정할 수 없다고 판단된 점, 신청인이 착화 과정에서 이 사건 제품


01 11
### 이 사건은 세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구에 대한 사례입니다.

살피건대, 피신청인은 세탁 전부터 이 사건 제품의 상태가 좋지 않았다고 주장


01 10
### 이 사건은 세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구에 대한 사례입니다.

나. 한국소비자원 신발제품심의위원회 심의 결과는 다음과 같다.
    신청인


01 11
### 이 사건은 세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구에 대한 사례입니다.

4 ● 2018 서비스·집단 분쟁조정 사례집 2 .  판   단 신청인은 피


01 10
### 이 사건은 세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구에 대한 사례입니다.

주 문 1. 신청인은 2018. 10. 16.까지 피신청인에게 이 사건 제품




### Generation

In [27]:
def format_docs(docs):
    """Document 객체에서 page_content 추출"""
    return "\n\n".join([d.page_content for d in docs])

retriever = vectorstore.as_retriever(search_kwargs={'k':5})

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "다음 컨텍스트를 참고하여 질문에 답하세요\n 컨텍스트에 없는 내용은 모른다 라고 답하세요\n\n컨텍스트:\n{context}"),
    ("human", "{query}")
])

# 질의 -> 벡터화 -> 가장 가까운 CHUNK 찾기  -> Document 객체 -> format_docs -> context -> llm context 기반으로 답변 정리

chain = {
            "context": retriever | format_docs,
            "query": RunnablePassthrough()
        } | rag_prompt | watson_llm | StrOutputParser()

response = chain.invoke(query)


### BM25 (Sparse Retrieval)
- 키워드 검색

In [30]:
bm25_retriever = BM25Retriever.from_documents(final_docs)
bm25_retriever.k = 5

docs = bm25_retriever.invoke("가죽 운동화 세탁 손해배상")

for doc in retriever_docs:
    print(doc.metadata['case_id'],doc.page_content[:100])
    print("="*50)


01 ### 이 사건은 세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구에 대한 사례입니다.

탁과실로 단정할 수 없다고 판단된 점, 신청인이 착화 과정에서 이 사건 제품
01 ### 이 사건은 세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구에 대한 사례입니다.

살피건대, 피신청인은 세탁 전부터 이 사건 제품의 상태가 좋지 않았다고 주장
01 ### 이 사건은 세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구에 대한 사례입니다.

나. 한국소비자원 신발제품심의위원회 심의 결과는 다음과 같다.
    신청인
01 ### 이 사건은 세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구에 대한 사례입니다.

4 ● 2018 서비스·집단 분쟁조정 사례집 2 .  판   단 신청인은 피
01 ### 이 사건은 세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구에 대한 사례입니다.

주 문 1. 신청인은 2018. 10. 16.까지 피신청인에게 이 사건 제품


- 시맨틱 검색 + 키워드 검색

In [31]:
ensemble_retriever = EnsembleRetriever(retrievers=[bm25_retriever, retriever], weights=[0.3,0.7])
docs = ensemble_retriever.invoke("가죽 운동화 세탁 손해배상")
for doc in docs:
    print(doc.metadata['case_id'], doc.page_content[:100])
    print("="*50)

01 ### 이 사건은 세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구에 대한 사례입니다.

주 문 1. 신청인은 2018. 10. 16.까지 피신청인에게 이 사건 제품
01 ### 이 사건은 세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구에 대한 사례입니다.

나. 한국소비자원 신발제품심의위원회 심의 결과는 다음과 같다.
    신청인
01 ### 이 사건은 세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구에 대한 사례입니다.

제1장 일 반 분 쟁 조 정  사 례 ( 서 비 스 ) 제1장 일반분쟁조정 
01 ### 이 사건은 세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구에 대한 사례입니다.

4 ● 2018 서비스·집단 분쟁조정 사례집 2 .  판   단 신청인은 피
01 ### 이 사건은 세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구에 대한 사례입니다.

탁과실로 단정할 수 없다고 판단된 점, 신청인이 착화 과정에서 이 사건 제품
01 ### 이 사건은 세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구에 대한 사례입니다.

살피건대, 피신청인은 세탁 전부터 이 사건 제품의 상태가 좋지 않았다고 주장


### SelfQuery

In [32]:

# 메타데이터 필드 정보 생성
metadata_field_info = [
    AttributeInfo(
        name="case_id", 
        description="사건번호", 
        type="string"
    ),
     AttributeInfo(
        name="title", 
        description="사건제목", 
        type="string"
    ),
     AttributeInfo(
        name="decision_date", 
        description="결정일자", 
        type="string"
    )
]

self_retriever = SelfQueryRetriever.from_llm(
    llm = watson_llm,
    vectorstore=vectorstore,
    document_contents="소비자 분쟁 사례",
    metadata_field_info = metadata_field_info,
    structured_query_translator=ChromaTranslator()
)

docs = self_retriever.invoke("32번 사례 보여줘")

for doc in docs:
    print(doc.metadata["case_id"])

32
32
32
32


일반 검색: k=5 
5 * 500chunk = 2500 자 전달 됨
2500 전달 대신에 질문과 관련있는 한두 문장만 찾기 => LLMChainExtractor

In [35]:
compressor = LLMChainExtractor.from_llm(watson_llm)
compression_retriever = (ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=ensemble_retriever
))

docs = compression_retriever.invoke("식당에서 분실된 신발에 대한 손해배상은 어떻게 결정되었나요?")

for doc in docs:
    print(doc.metadata["case_id"])

page_content='### 이 사건은 식당에서 분실된 신발에 대한 배상 요구에 대한 사례입니다.

다만, 신청인은 자신의 신발을 누구나 접근할 수 있는 개방된 신발장에 비치하면서 분 실 가능성이 있음을 충분히 예상할 수 있었고, 피신청인이 비치한 비닐봉지를 이용하 여 자신의 신발을 다른 신발과 구분하는 등 주의를 기울일 필요가 있었음에도 어떠한  조치도 취하지 아니하였으므로, 이러한 신청인의 부주의를 고려하여 피신청인의 책임 을 50%로 제한하기로 한다.' metadata={'producer': 'Acrobat Distiller 9.0.0 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2019-06-05T11:33:24+09:00', 'author': 'PC_A2', 'moddate': '2019-06-05T11:58:31+09:00', 'title': '식당에서 분실된 신발에 대한 배상 요구', 'source': './data/2018 서비스·집단 분쟁조정 사례집.pdf', 'total_pages': 200, 'page': 96, 'page_label': '97', 'case_id': '32', 'case_number': '2017일나273', 'decision_data': '2017.9.26.'}
32
page_content='### 이 사건은 식당에서 분실된 신발에 대한 배상 요구에 대한 사례입니다.

살피건대, 공중접객업자는 자기 또는 그 사용인이 고객으로부터 임치(任置)받은 물건의  보관에 관하여 주의를 게을리하지 아니하였음을 증명하지 아니하면 그 물건의 멸실 또 는 훼손으로 인한 손해를 배상할 책임이 있다.' metadata={'producer': 'Acrobat Distiller 9.0.0 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2019-06-05T11:33:24+09:00', '